# Tutorial 10: Soft Body Simulation for Robot Manipulation

## Why Simulate Soft Objects?

**Real robots don't just handle rigid objects.** They need to:
- 🍎 **Pick fruit** without bruising it
- 👕 **Fold laundry** and handle fabrics
- 🧸 **Grasp toys** that squish and deform
- 🥩 **Handle food** in kitchens and factories
- 🏥 **Manipulate tissue** in surgical robotics

If your robot only trains on rigid objects, it will **crush** soft things or **drop** them because it doesn't understand how they deform under grip forces.

This tutorial shows you how to simulate soft, deformable objects so your robot can learn to handle them properly.

## What You'll Learn

1. **Why soft body simulation matters** for robot manipulation
2. **Material parameters** — making objects squishy, firm, or rubbery
3. **Implicit integration** — stable physics even with stiff materials
4. **FEM meshes** — realistic volumetric deformation


## 1. Setup and Imports


In [1]:
import warp as wp
import numpy as np
import newton
from tqdm.notebook import trange

# Import from main newton package
from newton.solvers import SolverSoft

# Alias for tutorial consistency
SoftBodySolver = SolverSoft

print("✓ Loaded SolverSoft (as SoftBodySolver)")

# Initialize Warp
wp.init()
print(f"✓ Using device: {wp.get_device()}")
print(f"✓ Newton version: {newton.__version__}")


Warp 1.11.0.dev20251123 initialized:
   Git commit: 8b8f0b85ca54c0026574f834764e26615056aef6
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L40S" (44 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0.dev20251123
✓ Loaded SolverSoft (as SoftBodySolver)
✓ Using device: cuda:0
✓ Newton version: 0.1.3


## 2. The Object We're Simulating

For soft body simulation, we need a **tetrahedral mesh** — a 3D volume broken into tiny pyramids (tetrahedra). Unlike surface meshes used for rendering, volumetric meshes let us simulate how the *inside* of an object deforms.

**Why this matters for robots:**
- When a gripper squeezes a tomato, the whole volume deforms, not just the surface
- Pressure distributes through the material, affecting grip stability
- Different materials (foam vs. rubber vs. gel) respond very differently

We load a mesh from a `.mesh` file (Medit format) containing:
1. **Vertices** — 3D positions of all mesh points
2. **Tetrahedra** — volumetric elements connecting 4 vertices each


In [2]:
# Mesh file path
mesh_file = "spot.mesh"  # Tetrahedral mesh file (Medit format)

# Simulation parameters
initial_height = 1.0   # Drop height (meters)
mesh_scale = 1.0       # Scale factor for the mesh
mass = 2.0             # Mesh mass (kg)

# =============================================================================
# MATERIAL PROPERTIES — The Key to Realistic Soft Object Behavior
# =============================================================================
# These parameters determine how your object feels when a robot grabs it.
# Tuning these lets you simulate anything from a stress ball to a brick.

# Shear modulus (k_mu): How much the object resists shape change
#   - LOW  (1e3):  Jelly, very soft foam — deforms easily under grip
#   - MED  (1e5):  Rubber ball, stress toy — bouncy, springy
#   - HIGH (1e6):  Stiff rubber, firm plastic — barely deforms
k_mu = 1.0e6

# Bulk modulus (k_lambda): How much the object resists volume change
#   - LOW  (1e3):  Sponge — can be squeezed smaller
#   - MED  (1e5):  Foam rubber — some compressibility
#   - HIGH (1e6):  Water balloon — volume stays constant, just reshapes
k_lambda = 1.0e6

# Damping (k_damp): How quickly vibrations die out
#   - LOW  (0.1):  Bouncy like a rubber ball
#   - MED  (1.0):  Some bounce, settles quickly
#   - HIGH (10.0): Dead thud, like memory foam
k_damp = 1.0

# TRY THIS: Change k_mu to 1.0e4 to see a much softer, squishier object!

# Spring properties (for the mesh structure)
spring_ke = 1.0e5      # Spring stiffness
spring_kd = 1.0        # Spring damping

# Simulation settings
gravity = 9.81         # Gravity (m/s²)
fps = 60               # Frames per second
substeps = 5           # Physics substeps per frame
frame_dt = 1.0 / fps   # Time per frame
sim_dt = frame_dt / substeps  # Physics timestep

print(f"Physics timestep: {sim_dt*1000:.2f} ms")
print(f"Substeps per frame: {substeps}")
print(f"\nMaterial: k_mu={k_mu:.0e}, k_lambda={k_lambda:.0e}, k_damp={k_damp}")


Physics timestep: 3.33 ms
Substeps per frame: 5

Material: k_mu=1e+06, k_lambda=1e+06, k_damp=1.0


In [3]:
def load_medit_mesh(filename):
    """Load a Medit .mesh file and extract vertices and tetrahedra."""
    vertices = []
    tetrahedra = []
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line == "Vertices":
            i += 1
            num_verts = int(lines[i].strip())
            i += 1
            for _ in range(num_verts):
                parts = lines[i].strip().split()
                vertices.append([float(parts[0]), float(parts[1]), float(parts[2])])
                i += 1
        elif line == "Tetrahedra":
            i += 1
            num_tets = int(lines[i].strip())
            i += 1
            for _ in range(num_tets):
                parts = lines[i].strip().split()
                # Medit uses 1-based indexing, convert to 0-based
                tetrahedra.append([int(parts[0])-1, int(parts[1])-1, int(parts[2])-1, int(parts[3])-1])
                i += 1
        else:
            i += 1
    
    return np.array(vertices), np.array(tetrahedra)

def normalize_mesh(vertices):
    """Normalize mesh vertices by scaling to unit size."""
    extents = vertices.max(axis=0) - vertices.min(axis=0)
    max_extent = np.max(extents)
    scale_factor = 1.0 / max_extent if max_extent > 0 else 1.0
    vertices_scaled = vertices * scale_factor
    return vertices_scaled, scale_factor, extents

# Load mesh from file
print(f"Loading mesh from: {mesh_file}")
vertices_raw, tetras = load_medit_mesh(mesh_file)
print(f"Raw mesh: {len(vertices_raw)} vertices, {len(tetras)} tetrahedra")

# Normalize and scale the mesh
vertices_normalized, auto_scale, extents = normalize_mesh(vertices_raw)
vertices = (vertices_normalized * mesh_scale).astype(np.float32)

# Flatten tetrahedra indices for Newton
indices = tetras.flatten().astype(np.int32)

print(f"\nMesh statistics:")
print(f"  Vertices: {len(vertices)}")
print(f"  Tetrahedra: {len(tetras)}")
print(f"  Original extents: {extents}")
print(f"  Scaled size: {mesh_scale:.2f}m")


Loading mesh from: spot.mesh
Raw mesh: 4273 vertices, 16471 tetrahedra

Mesh statistics:
  Vertices: 4273
  Tetrahedra: 16471
  Original extents: [1.71723697 0.94255724 1.69025766]
  Scaled size: 1.00m


## 3. Building the Simulation Scene

We use Newton's `ModelBuilder` to construct the world:
- **Ground plane** — with contact and friction properties
- **Soft mesh** — our deformable object with material parameters

In a robotics scenario, you'd also add:
- Robot arm geometry (as rigid bodies)
- Gripper surfaces (with appropriate friction for grasping)
- Other objects in the scene


In [4]:
# Create the Newton model builder
builder = newton.ModelBuilder()

# Add ground plane at Z=0 with contact properties
builder.add_ground_plane(
    cfg=newton.ModelBuilder.ShapeConfig(
        ke=5e5,   # Contact stiffness
        kd=1e3,   # Contact damping
        kf=1e4,   # Friction stiffness
        mu=0.5    # Friction coefficient
    )
)

# Track particle start index before adding soft mesh
start_particle = len(builder.particle_q)

# Add the soft doll mesh positioned above ground
# Z-axis is up in Newton's coordinate system
builder.add_soft_mesh(
    pos=wp.vec3(0.0, 0.0, initial_height),  # Start above ground
    rot=wp.quat_identity(),
    vel=wp.vec3(0.0, 0.0, 0.0),             # Start at rest
    vertices=vertices,
    indices=indices,
    scale=1.0,
    density=mass,
    k_mu=k_mu,
    k_lambda=k_lambda,
    k_damp=k_damp,
)

# Add springs between mesh vertices for stability
# SolverSoft uses springs for its implicit integration system matrix
# spring_ke and spring_kd defined in parameters cell
num_tets = len(tetras)
vertex_positions = np.array(vertices)

# Track added springs to avoid duplicates
added_springs = set()

for t in range(num_tets):
    tet_indices = [indices[t * 4 + k] for k in range(4)]
    # All 6 edges of the tetrahedron
    edges = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
    for ei, ej in edges:
        i_local, j_local = tet_indices[ei], tet_indices[ej]
        # Ensure consistent ordering to avoid duplicates
        if i_local > j_local:
            i_local, j_local = j_local, i_local
        
        spring_key = (i_local, j_local)
        if spring_key not in added_springs:
            added_springs.add(spring_key)
            
            # Compute rest length from initial vertex positions
            p0 = vertex_positions[i_local]
            p1 = vertex_positions[j_local]
            rest_length = float(np.linalg.norm(p1 - p0))
            
            # Add spring with particle indices (offset by start_particle)
            builder.add_spring(
                start_particle + i_local, 
                start_particle + j_local, 
                spring_ke, 
                spring_kd, 
                rest_length
            )

print(f"Added {len(added_springs)} springs")

# Finalize the model
model = builder.finalize()

# Set gravity (Z is up, gravity pulls down)
model.gravity = wp.vec3(0.0, 0.0, -gravity)

# Contact parameters (must match working example!)
model.soft_contact_ke = 5.0e4   # Contact stiffness
model.soft_contact_kd = 500.0   # Contact damping (NOT 2000 - causes explosion!)
model.soft_contact_kf = 5.0e4   # Friction stiffness
model.soft_contact_mu = 0.9     # Friction coefficient

# Particle constraint parameters (needed for stability!)
model.particle_ke = 1.0e5
model.particle_kd = 1.0

# Set particle radius for contact detection
model.particle_radius = wp.array(
    np.full(model.particle_count, 0.008),
    dtype=wp.float32,
    device=model.device
)

print(f"\nModel created:")
print(f"  Particles: {model.particle_count}")
print(f"  Springs: {model.spring_count}")
print(f"  Tetrahedra: {model.tet_count}")
print(f"  Triangles: {model.tri_count}")


Added 23433 springs

Model created:
  Particles: 4273
  Springs: 23433
  Tetrahedra: 16471
  Triangles: 5380


## 4. The Physics Solver: Why Implicit Integration Matters for Robots

**The Problem:** Robot controllers run at fixed rates (e.g., 100 Hz). Simple physics simulations explode when materials are stiff because you need tiny timesteps (10,000+ Hz). That's way too slow for real-time robot training.

**The Solution:** Implicit integration lets us take big, stable timesteps even with stiff materials.

### How It Works

Every timestep, we solve:

$$\mathbf{A} \cdot \Delta\mathbf{v} = \mathbf{b}$$

Where:
- $\mathbf{A}$ = system matrix (describes how particles are connected)
- $\Delta\mathbf{v}$ = velocity changes we're solving for
- $\mathbf{b}$ = forces × timestep

The matrix $\mathbf{A}$ combines:
- **Mass** ($\mathbf{M}$) — heavier particles are harder to move
- **Damping** ($\mathbf{D}$) — energy loss from internal friction
- **Stiffness** ($\mathbf{K}$) — resistance to deformation

**Available solvers:**
- **BiCGStab** — default, works for everything
- **CG** — fastest, for symmetric systems
- **GMRES/CR** — for tricky edge cases


In [5]:
# Create the SoftBodySolver with implicit integration
solver = SoftBodySolver(
    model=model,
    dt=sim_dt,
    mass=mass,
    solver_type="bicgstab"  # Options: "cg", "bicgstab", "gmres", "cr"
)

# Create state objects for time integration
state_0 = model.state()  # Current state
state_1 = model.state()  # Next state

# Create control object
control = model.control()

print("\nSolver ready!")
print(f"  Solver: SoftBodySolver (Implicit Integration)")
print(f"  Linear solver: {solver.solver_type}")
print(f"  Particles: {model.particle_count}")
print(f"  Timestep: {sim_dt*1000:.2f} ms")


Module newton._src.solvers.soft.kernels e3ced69 load on device 'cuda:0' took 3303.17 ms  (compiled)
Module warp.sparse 9cae4f0 load on device 'cuda:0' took 376.12 ms  (compiled)

Solver ready!
  Solver: SoftBodySolver (Implicit Integration)
  Linear solver: bicgstab
  Particles: 4273
  Timestep: 3.33 ms


## Understanding Soft Body Physics (For Robotics)

### Why Robots Need to Understand Deformation

When a robot gripper squeezes an object:
1. **Rigid object** → The gripper feels resistance, object doesn't change shape
2. **Soft object** → The object deforms, contact area grows, grip becomes more stable

This matters because:
- **Grip force sensing**: A soft object "gives" before building resistance
- **Stable grasping**: Deformation creates larger contact patches = more friction
- **Damage prevention**: Know when you're squeezing too hard before you crush it

### The Physics Model

We represent the soft object as **particles connected by springs**:

| Property | Symbol | Meaning |
|----------|--------|---------|
| Position | $\mathbf{x}$ | Where each particle is |
| Velocity | $\mathbf{v}$ | How fast it's moving |
| Mass | $m$ | How heavy it is |

Newton's 2nd law governs motion:

$$\ddot{\mathbf{x}} = \frac{\mathbf{F}}{m}$$

The forces $\mathbf{F}$ include:
- **Internal springs** — try to restore the object's shape
- **Gravity** — pulls everything down
- **Damping** — energy loss (like internal friction)
- **Contact forces** — from gripper, table, or other objects

### The Simple Way: Explicit Euler (Why It Doesn't Work Well)

The obvious approach: look at where you are now, figure out the forces, then move.

$$\text{new velocity} = \text{old velocity} + \text{timestep} \times \frac{\text{force}}{\text{mass}}$$

$$\text{new position} = \text{old position} + \text{timestep} \times \text{new velocity}$$

Or in math notation:

$$\mathbf{v}_{n+1} = \mathbf{v}_n + \Delta t \cdot \frac{\mathbf{f}(\mathbf{x}_n, \mathbf{v}_n)}{m}$$

$$\mathbf{x}_{n+1} = \mathbf{x}_n + \Delta t \cdot \mathbf{v}_{n+1}$$

**The Problem:** Springs are stiff! If you take a big timestep, the spring overshoots, then overcorrects, then explodes. You need tiny timesteps (like 0.0001 seconds) which is super slow.

### The Smart Way: Implicit Euler (What We Actually Use)

Here's the trick: instead of asking "where will I be?", we ask "where do I need to end up so that the forces there push me to exactly that spot?"

It's like catching a ball — you don't reach for where the ball *is*, you reach for where it *will be*.

$$\mathbf{v}_{n+1} = \mathbf{v}_n + \Delta t \cdot \frac{\mathbf{f}(\mathbf{x}_{n+1}, \mathbf{v}_{n+1})}{m}$$

But wait... we need to know the future position to calculate the force, but we need the force to find the future position! 🤯

#### The Math Trick: Linearization

We approximate: "the force at the new position is roughly the force now, plus how much it changes as we move."

$$\mathbf{f}_{\text{new}} \approx \mathbf{f}_{\text{now}} + \mathbf{K} \cdot \Delta\mathbf{x} + \mathbf{D} \cdot \Delta\mathbf{v}$$

Where:
- $\mathbf{K}$ = **stiffness matrix** (how much force changes when position changes)
- $\mathbf{D}$ = **damping matrix** (how much force changes when velocity changes)
- $\Delta\mathbf{x}$ = change in position
- $\Delta\mathbf{v}$ = change in velocity

#### Solving the System

Since $\Delta\mathbf{x} = \Delta t \cdot \Delta\mathbf{v}$ (position change = timestep × velocity change), we can rewrite everything as one equation:

$$\mathbf{A} \cdot \Delta\mathbf{v} = \mathbf{b}$$

Where:
- $\mathbf{A} = \mathbf{M} - \Delta t \cdot \mathbf{D} - \Delta t^2 \cdot \mathbf{K}$ (the "system matrix")
- $\mathbf{b} = \Delta t \cdot \mathbf{f}$ (the forces × timestep)
- $\Delta\mathbf{v}$ = what we're solving for (velocity change)

This is just like solving $Ax = b$ in algebra, but with thousands of particles!

#### Why Implicit is Better

| Method | Timestep Limit | Result |
|--------|---------------|--------|
| **Explicit** | Must be tiny (~0.0001s) | Slow, might explode 💥 |
| **Implicit** | Can be large (~0.01s) | Fast, always stable ✓ |

Implicit can take timesteps **100× larger** because it "looks ahead" to where things will be.

### Force Models (What Makes Things Squish)

#### Springs: Measuring Deformation

Springs connect particles and resist stretching — **this is how robots can sense deformation**.

```
    particle i ●────────● particle j
              <-- spring -->
```

**Force formula:**
$$\mathbf{f} = \left( k \cdot (\ell - L_0) + k_d \cdot \dot{\ell} \right) \cdot \hat{\mathbf{d}}$$

| Symbol | Meaning | Robot Relevance |
|--------|---------|-----------------|
| $\ell$ | Current length | Measurable deformation |
| $L_0$ | Rest length | Baseline shape |
| $k$ | Stiffness | Material property |
| $\varepsilon = \frac{\ell - L_0}{L_0}$ | **Strain** | **Feedback for grip control** |

**For robots**: By monitoring strain ($\varepsilon$), a gripper can detect when it's deforming an object and adjust force accordingly.

#### FEM Tetrahedra: 3D Volume Deformation

For solid objects, we use tetrahedra (4-cornered 3D elements) to model volumetric deformation.

**Deformation gradient** $\mathbf{F}$: Measures how much the tetrahedron has squished/stretched compared to its rest shape.

**Stable Neo-Hookean stress** (used in the code):
$$\mathbf{P} = \mu \cdot \mathbf{F} \cdot \left(1 - \frac{1}{I_C + 1}\right)$$

Where $I_C = \|\mathbf{F}\|^2$ measures total deformation.

**Volume preservation:**
$$\mathbf{f}_{\text{volume}} = \lambda \cdot (J - 1) \cdot \nabla J$$

Where $J = \det(\mathbf{F})$ is the volume ratio:
- $J = 1$ → same volume
- $J < 1$ → compressed (like squeezing a sponge)
- $J > 1$ → expanded

**For robots**: The $\lambda$ parameter controls compressibility. High $\lambda$ means the object reshapes but doesn't shrink (like a water balloon). Low $\lambda$ allows compression (like a sponge).

### How the Code Solves It

The system matrix $\mathbf{A}$ is **sparse** — most entries are zero because each particle only connects to a few neighbors. We store it efficiently in **BSR format** (Block Sparse Row):

- Each spring creates 4 blocks of 3×3 matrices
- Blocks connect particle pairs: (i,i), (j,j), (i,j), (j,i)

**Linear Solvers** (pick one):
| Solver | Best For | Speed |
|--------|----------|-------|
| **BiCGStab** | General use (default) | Fast |
| **CG** | Symmetric matrices | Fastest |
| **GMRES** | Tricky matrices | Reliable |
| **CR** | Ill-conditioned | Robust |

With our mesh of 4,273 particles (12,819 variables!), the solver typically converges in just 3-10 iterations.

### The Bottom Line

| | Explicit | Implicit (what we use) |
|---|---|---|
| **Timestep** | 0.0001s | 0.01s |
| **Speed** | Slow | 100× faster |
| **Stability** | Might explode | Always stable |

### Applications in Robotics

#### 1. Training Manipulation Policies

Soft body simulation lets you train robots in simulation before deploying on real hardware:

| Application | Material Type | Why It Matters |
|------------|---------------|----------------|
| **Fruit picking** | Soft, fragile | Learn force limits to avoid bruising |
| **Surgical robots** | Tissue-like | Practice without harming patients |
| **Textile handling** | Cloth, fabric | Learn folding and grasping |
| **Food preparation** | Various | Handle dough, meat, vegetables |

#### 2. Grip Force Estimation

The strain in springs tells you how much the object is deforming:

$$\varepsilon = \frac{\ell - L_0}{L_0}$$

If $\varepsilon = 0.1$, the spring is stretched 10%. A robot can use this to:
- Detect when grip is secure (deformation has stabilized)
- Know when it's squeezing too hard (excessive strain)
- Adjust force in real-time based on material response

#### 3. Real-Time Control Compatibility

Because implicit integration runs at **practical controller rates** (50-100 Hz), you can:
- Use the same timesteps as your real robot controller
- Train policies that transfer directly to hardware
- Run multiple parallel simulations for RL training

### References

[1] Baraff & Witkin, "[Large Steps in Cloth Simulation](https://www.cs.cmu.edu/~baraff/papers/sig98.pdf)," SIGGRAPH 1998 — Implicit integration for soft bodies

[2] Smith, Goes, Kim, "[Stable Neo-Hookean Flesh Simulation](https://graphics.pixar.com/library/StableElasticity/paper.pdf)," SIGGRAPH 2018 — The FEM material model

## 5. Simulation Functions

The simulation loop performs these steps each frame:
1. Clear accumulated forces
2. Detect collisions between particles and shapes
3. Compute and apply all forces (springs, FEM, gravity, contacts)
4. Solve the implicit system for velocity updates
5. Update positions


In [6]:
def simulate_frame():
    """Run physics simulation for one frame (multiple substeps)."""
    global state_0, state_1
    
    for _ in range(substeps):
        # Clear accumulated forces
        state_0.clear_forces()
        
        # Collision detection - returns contacts
        contacts = model.collide(state=state_0)
        
        # Physics integration step using SolverSoft
        # The solver uses implicit integration with sparse matrix solvers
        solver.step(
            state_in=state_0,
            state_out=state_1,
            control=control,
            contacts=contacts,
            dt=sim_dt
        )
        
        # Swap states (next becomes current)
        state_0, state_1 = state_1, state_0

print("Simulation function defined!")


Simulation function defined!


## 6. Visualization with Rerun

Newton integrates with [Rerun](https://rerun.io/) for real-time 3D visualization.


In [7]:
# Create the Rerun viewer
# keep_historical_data=True allows time scrubbing through the simulation
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)

# Set the model (logs static geometry)
viewer.set_model(model)

print("✓ Rerun viewer initialized!")


Module newton._src.viewer.kernels 1205f75 load on device 'cuda:0' took 1277.81 ms  (compiled)
✓ Rerun viewer initialized!


In [8]:
# Viewer is ready - will be used in simulation loop
print("Viewer ready for simulation!")


Viewer ready for simulation!


## 7. Run the Simulation

Let's drop the soft object and watch how it deforms on impact — the kind of behavior a robot needs to understand for safe manipulation.


In [9]:
# Simulation parameters
num_frames = 300
sim_time = 0.0

# Tracking variables
bounce_count = 0
was_falling = True
max_height = 0.0

print(f"🎎 Dropping doll from height: {initial_height}m")
print(f"   Running {num_frames} frames ({num_frames/fps:.1f} seconds)...\n")

for frame in trange(num_frames, desc="Simulating"):
    # Run physics
    simulate_frame()
    
    # Get doll position (center of mass)
    positions = state_0.particle_q.numpy()
    center = positions.mean(axis=0)
    height = center[2]  # Z is up
    
    # Track bounces (velocity sign change)
    velocities = state_0.particle_qd.numpy()
    avg_vel_z = velocities.mean(axis=0)[2]
    
    is_falling = avg_vel_z < 0
    if was_falling and not is_falling and height < initial_height * 0.8:
        bounce_count += 1
        print(f"   🎎 Bounce #{bounce_count} at height {height:.2f}m")
    was_falling = is_falling
    
    max_height = max(max_height, height)
    
    # Log to viewer
    contacts = model.collide(state=state_0)
    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()
    
    sim_time += frame_dt

print(f"\n🎎 Simulation complete!")
print(f"   Total bounces: {bounce_count}")
print(f"   Max height reached: {max_height:.2f}m")


🎎 Dropping doll from height: 1.0m
   Running 300 frames (5.0 seconds)...



Simulating:   0%|          | 0/300 [00:00<?, ?it/s]

Module newton._src.geometry.kernels 3d38582 load on device 'cuda:0' took 3689.67 ms  (compiled)
Module newton._src.solvers.soft.particles f6d3c65 load on device 'cuda:0' took 332.68 ms  (compiled)
Module map_add d16df81 load on device 'cuda:0' took 165.46 ms  (compiled)
Module map_mul 1a56e01 load on device 'cuda:0' took 158.54 ms  (compiled)
Module warp.optim.linear 9540452 load on device 'cuda:0' took 515.69 ms  (compiled)
Module warp.optim.linear cb25921 load on device 'cuda:0' took 240.97 ms  (compiled)
Module warp.optim.linear 58c84f3 load on device 'cuda:0' took 264.49 ms  (compiled)
Module warp._src.sparse.dyn.bsr_mv_kernel_0d4f3dc9 0adad09 load on device 'cuda:0' took 150.46 ms  (compiled)
Module warp.optim.linear 454adcd load on device 'cuda:0' took 310.97 ms  (compiled)
Module warp.optim.linear f5a5d06 load on device 'cuda:0' took 350.97 ms  (compiled)
Module warp.optim.linear 1e40c09 load on device 'cuda:0' took 393.61 ms  (compiled)
Module warp.optim.linear 5b24459 load on 

In [10]:
# Display the viewer - use time scrubbing to replay the simulation
viewer


HTML(value='<div id="e6b02933-18a8-4be5-8234-bb9fb632b44d"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

## 8. What's Next?


## Summary: What You Learned

### Key Concepts for Robotics

1. **Why Soft Bodies Matter**: Real-world robots handle non-rigid objects constantly. Simulating deformable materials lets you:
   - Train manipulation policies safely
   - Test grip force control before damaging real objects
   - Handle varied materials (fruit, fabric, foam, tissue)

2. **Material Properties**: The key parameters that define "how squishy":
   | Parameter | Controls | Robot Application |
   |-----------|----------|-------------------|
   | `k_mu` | Shape resistance | How much object deforms under grip |
   | `k_lambda` | Volume resistance | Whether object compresses or just reshapes |
   | `k_damp` | Energy loss | How quickly deformations settle |

3. **Implicit Integration**: Enables real-time simulation at robot controller rates (50-100 Hz) instead of requiring unrealistic 10,000 Hz updates.

4. **Deformation Sensing**: Spring strain ($\varepsilon$) provides feedback about how much an object is being deformed — useful for grip force estimation.

### Try These Experiments

**Material variations** — Change `k_mu` in the parameters cell:
- `k_mu = 1.0e4` → Very soft (like a stress ball)
- `k_mu = 1.0e5` → Medium (like rubber)  
- `k_mu = 1.0e6` → Stiff (like hard foam)

**Damping variations** — Change `k_damp`:
- `k_damp = 0.1` → Very bouncy
- `k_damp = 10.0` → Dead thud (like memory foam)

### Next Steps

- Add a robot gripper to the scene and simulate grasping
- Create a dataset of varied materials for training
- Implement force feedback based on deformation strain
- Try cloth simulation for fabric handling tasks


---
*Tutorial 10: Soft Body Simulation for Robot Manipulation*

**Key Takeaway**: Robots that only train on rigid objects will fail when handling soft materials. This tutorial gave you the tools to simulate deformable objects for manipulation training.